# MPO AutoSat Workflow

This notebook runs a full MPO workflow in three stages:
1. Warmup (with 2 render videos + reward summary)
2. Training (with 3 sample render videos + reward summary)
3. Test loop (best 2 runs by total reward)

Artifacts are stored under `backend/autonomous_control/models/runs/`.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import json
import sys

import numpy as np
# Resolve project/backend roots from notebook location.
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd.parent, _cwd.parent.parent, _cwd.parent.parent.parent]
project_root = None
for p in _candidates:
    if (p / "backend").exists():
        project_root = p
        break
if project_root is None:
    raise RuntimeError("Could not resolve project root containing backend/.")

backend_root = project_root / "backend"
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

from autonomous_control.config.randomness import RandomnessConfig, apply_global_seed, derive_seed
from autonomous_control.controller_agent import MPOAgent
from autonomous_control.mpo_config import MPOConfig
from autonomous_control.training_runtime import make_attitude_control_env, run_episode
from environment_definition.constants import RenderMode
from environment_definition.mission_profiles.mission_1_random_fl import sample_satellite_altitude
from render.render_main import render_from_series
from utils.ml_training.ml_training_utils import create_run_dir, init_run_markdown, append_run_markdown_event

SEED = 7
VIDEOS_PER_CELL = 1
RNG_CFG = RandomnessConfig(seed=SEED)
apply_global_seed(RNG_CFG)
SATELLITE_ALTITUDE = sample_satellite_altitude(seed=derive_seed(SEED, "mission_altitude"))

RUN_ID = f"nb-mpo-{datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S')}"
RUN_DIR = create_run_dir(run_id=RUN_ID)

init_run_markdown(
    RUN_DIR,
    title="Notebook MPO Workflow",
    metadata={
        "seed": SEED,
        "sampled_altitude_km": float(SATELLITE_ALTITUDE.to("km").magnitude),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    },
)

print(f"RUN_DIR: {RUN_DIR}")
print(f"Sampled altitude: {SATELLITE_ALTITUDE}")

warmup_episode_count = 10
train_episode_count = 5
test_episode_count = 1

print(f"""
##########################################
##########################################
Warmup episodes: {warmup_episode_count}
#################################
Training ep:     {train_episode_count}
#################################
Test ep:         {test_episode_count}
##########################################
##########################################
""")


/home/cedric/miniforge3/envs/auto-sat/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RUN_DIR: /home/cedric/code/auto-sat-control/backend/autonomous_control/models/nb-mpo-2026-05-04_13-06-12
Sampled altitude: 528.758 km

##########################################
##########################################
Warmup episodes: 10
#################################
Training ep:     5
#################################
Test ep:         1
##########################################
##########################################



In [2]:
@dataclass
class EpisodeArtifact:
    stage: str
    index: int
    total_reward: float
    average_reward: float
    steps: int
    video_path: Path

from utils.mpo_notebook_video import display_mpo_video, init_mpo_video_cell, log_exported_video

def _export_render_video(*, simulation_series, out_path: Path) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    result = render_from_series(
        simulation_series=simulation_series,
        render_mode=RenderMode.EXPORT,
        output_path=out_path,
    )
    if result is None:
        raise RuntimeError("Render export did not return output path.")
    if not out_path.exists() or out_path.stat().st_size <= 0:
        raise RuntimeError(f"Render export missing/empty video: {out_path}")
    log_exported_video(out_path, tag="after_export")
    return out_path


def _episode_metrics(result) -> tuple[float, float, int]:
    total = float(result.episode_return)
    steps = int(result.steps)
    avg = total / max(1, steps)
    return total, avg, steps


def _write_csv(path: Path, rows: list[dict[str, object]]) -> None:
    if not rows:
        return
    import csv

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

def _display_video(path: Path, width: int = 680) -> None:
    display_mpo_video(path, width=width)


init_mpo_video_cell()


## Warmup Stage

Runs 2 warmup episodes and exports 2 rendered videos.
Also records total and average rewards.


In [3]:
warmup_env = make_attitude_control_env()
warmup_cfg = MPOConfig(warmup_episodes=0)
warmup_agent = MPOAgent(warmup_env, config=warmup_cfg)

warmup_artifacts: list[EpisodeArtifact] = []
warmup_rows: list[dict[str, object]] = []

for i in range(VIDEOS_PER_CELL):
    result = run_episode(
        warmup_env,
        warmup_agent,
        mode="warmup",
        train_updates_per_step=0,
        warmup_controller="random",
        satellite_altitude=SATELLITE_ALTITUDE,
        np_rng=np.random.default_rng(derive_seed(SEED, "nb_warmup", i)),
    )
    total, avg, steps = _episode_metrics(result)
    video_path = RUN_DIR / f"warmup_{i+1:02d}.mp4"
    _export_render_video(simulation_series=result.simulation_series, out_path=video_path)

    warmup_artifacts.append(
        EpisodeArtifact(
            stage="warmup",
            index=i,
            total_reward=total,
            average_reward=avg,
            steps=steps,
            video_path=video_path,
        )
    )
    warmup_rows.append(
        {
            "stage": "warmup",
            "episode": i + 1,
            "total_reward": total,
            "average_reward": avg,
            "steps": steps,
            "video": str(video_path),
        }
    )
    append_run_markdown_event(
        RUN_DIR,
        heading=f"Warmup episode {i + 1}",
        payload={
            "total_reward": f"{total:.6f}",
            "average_reward": f"{avg:.6f}",
            "steps": steps,
            "video": str(video_path),
        },
    )

warmup_csv = RUN_DIR / "warmup_metrics.csv"
_write_csv(warmup_csv, warmup_rows)
print(f"Warmup summary written: {warmup_csv}")
for row in warmup_rows:
    print(row)


Using device: cuda


/home/cedric/code/auto-sat-control/backend/simulation/stepper.py:130: UserWarning: controller_update_interval is not an integer multiple of simulation_timestep; using nearest multiple.
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(


[run_episode] start mode=warmup max_steps=1883            
warmup episode:   0%|          | 0/1883 [00:00<?, ?step/s]

'Step 0:'

'  Action (Nm): -0.022279'

...

warmup episode:  98%|█████████▊| 1836/1883 [00:03<00:00, 626.25step/s, reward=-100.00000, steps=1830]

'Step 1882:'

'  Action (Nm): -0.007874'

array([-2.6477158 ,  0.01408728,  1.9854541 ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ], dtype=float32)

[run_episode] end mode=warmup steps=1883 total_reward=nan avg_reward=nan                             
[mpo_video:after_export] warmup_01.mp4 (10754062 bytes)
Warmup summary written: /home/cedric/code/auto-sat-control/backend/autonomous_control/models/nb-mpo-2026-05-04_13-06-12/warmup_metrics.csv
{'stage': 'warmup', 'episode': 1, 'total_reward': nan, 'average_reward': nan, 'steps': 1883, 'video': '/home/cedric/code/auto-sat-control/backend/autonomous_control/models/nb-mpo-2026-05-04_13-06-12/warmup_01.mp4'}


In [4]:
for item in warmup_artifacts:
    #_display_video(item.video_path)
    pass


## Training Stage

Runs a short training loop, stores run/action artifacts, and exports 3 sample render videos.


In [5]:


train_env = make_attitude_control_env()
train_cfg = MPOConfig(warmup_episodes=0)
train_agent = MPOAgent(train_env, config=train_cfg)

train_results: list[dict[str, object]] = []
train_rows: list[dict[str, object]] = []

for ep in range(train_episode_count):
    result = run_episode(
        train_env,
        train_agent,
        mode="train",
        train_updates_per_step=1,
        satellite_altitude=SATELLITE_ALTITUDE,
        np_rng=np.random.default_rng(derive_seed(SEED, "nb_train", ep)),
    )
    train_results.append(result)
    total, avg, steps = _episode_metrics(result)
    train_rows.append(
        {
            "stage": "train",
            "episode": ep + 1,
            "total_reward": total,
            "average_reward": avg,
            "steps": steps,
        }
    )



# Persist training metrics and sampled actions for later warmup reuse experiments.
train_csv = RUN_DIR / "training_metrics.csv"
_write_csv(train_csv, train_rows)

buffer_count = int(len(train_agent.buffer))
actions_np = train_agent.buffer.actions[:buffer_count].copy()
obs_np = train_agent.buffer.obs[:buffer_count].copy()
np.savez_compressed(RUN_DIR / "training_samples.npz", actions=actions_np, obs=obs_np)

# Export configurable number of sample training videos.
sample_count = min(VIDEOS_PER_CELL, train_episode_count)
sample_indices = list(range(sample_count))
train_video_paths: list[Path] = []
for i, idx in enumerate(sample_indices, start=1):
    result = train_results[idx]
    video_path = RUN_DIR / f"train_sample_{i:02d}_ep{idx+1:02d}.mp4"
    _export_render_video(simulation_series=result.simulation_series, out_path=video_path)
    train_video_paths.append(video_path)

print(f"Training summary written: {train_csv}")
print(f"Training samples saved: {RUN_DIR / 'training_samples.npz'}")
for row in train_rows:
    print(row)
for p in train_video_paths:
    #_display_video(p)
    pass


Using device: cuda
[run_episode] start mode=train max_steps=1883            
train episode:   0%|          | 0/1883 [00:00<?, ?step/s]

'Step 0:'

'  Action (Nm): -0.091373'

array([-1.9859221e+00, -1.1708903e-03,  1.1565790e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
│   │   1.0000000e+00,  1.0000000e+00,  1.0000000e+00], dtype=float32)

train episode:   5%|▍         | 92/1883 [00:00<00:06, 260.44step/s, reward=-100.00000, steps=90]

...

train episode:  58%|█████▊    | 1095/1883 [00:14<00:12, 64.67step/s, reward=-100.00000, steps=1090]

'Step 1100:'

'  Action (Nm): -0.100000'

array([-0.11939659, -0.03754001,  1.6410437 ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
│   │   0.        ,  0.        ,  0.        ], dtype=float32)

ValueError: Expected parameter loc (Tensor of shape (256, 1)) of distribution Normal(loc: torch.Size([256, 1]), scale: torch.Size([256, 1])) to satisfy the constraint Real(), but found invalid values:
tensor([[nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
    

## Test Stage

Evaluates trained MPO policy, ranks by total reward, and exports the best 2 render videos.


In [ ]:
test_env = make_attitude_control_env()
# Reuse trained policy parameters in a fresh env adapter.
test_agent = train_agent

test_episode_count = 1
test_rows: list[dict[str, object]] = []
test_results = []

for ep in range(test_episode_count):
    result = run_episode(
        test_env,
        test_agent,
        mode="test",
        train_updates_per_step=0,
        satellite_altitude=SATELLITE_ALTITUDE,
        np_rng=np.random.default_rng(derive_seed(SEED, "nb_test", ep)),
    )
    total, avg, steps = _episode_metrics(result)
    test_rows.append(
        {
            "stage": "test",
            "episode": ep + 1,
            "total_reward": total,
            "average_reward": avg,
            "steps": steps,
        }
    )
    test_results.append(result)

# Rank by total reward; tie-break by average reward.
ranked = sorted(
    zip(test_rows, test_results),
    key=lambda item: (float(item[0]["total_reward"]), float(item[0]["average_reward"])),
    reverse=True,
)
best_ranked = ranked[:VIDEOS_PER_CELL]

best_rows: list[dict[str, object]] = []
for rank_idx, (row, result) in enumerate(best_ranked, start=1):
    out_path = RUN_DIR / f"test_best_{rank_idx:02d}_ep{int(row['episode']):02d}.mp4"
    _export_render_video(simulation_series=result.simulation_series, out_path=out_path)
    row_with_video = dict(row)
    row_with_video["rank"] = rank_idx
    row_with_video["video"] = str(out_path)
    best_rows.append(row_with_video)

all_test_csv = RUN_DIR / "test_metrics.csv"
best_test_csv = RUN_DIR / "best_test_metrics.csv"
_write_csv(all_test_csv, test_rows)
_write_csv(best_test_csv, best_rows)

print(f"Test summary written: {all_test_csv}")
print(f"Best run summary written: {best_test_csv}")
for row in best_rows:
    print(row)
for row in best_rows:
    #_display_video(Path(row["video"]))
    pass


In [ ]:
def _aggregate(rows: list[dict[str, object]]) -> dict[str, float]:
    totals = [float(r["total_reward"]) for r in rows]
    avgs = [float(r["average_reward"]) for r in rows]
    return {
        "episodes": float(len(rows)),
        "total_reward_mean": float(np.mean(totals)) if totals else 0.0,
        "total_reward_std": float(np.std(totals)) if totals else 0.0,
        "avg_reward_mean": float(np.mean(avgs)) if avgs else 0.0,
    }

summary = {
    "warmup": _aggregate(warmup_rows),
    "train": _aggregate(train_rows),
    "test": _aggregate(test_rows),
    "best_test": _aggregate(best_rows),
}
summary_path = RUN_DIR / "summary_metrics.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
append_run_markdown_event(RUN_DIR, heading="Notebook summary", payload={"summary_json": str(summary_path)})
print(json.dumps(summary, indent=2))
print(f"Summary JSON: {summary_path}")


## Appendix - Manual Notebook Experience Gate

Answer Y/N for each stage. Any `N` fails this notebook run.


In [ ]:
questions = {
    "warmup": "Warmup output/video quality to your liking? (Y/N): ",
    "training": "Training outputs and sampled videos to your liking? (Y/N): ",
    "test": "Best-run test videos/metrics to your liking? (Y/N): ",
    "summary": "Overall notebook experience to your liking? (Y/N): ",
}

responses: dict[str, str] = {}
for key, prompt in questions.items():
    ans = input(prompt).strip().upper()
    responses[key] = ans

failed = [k for k, v in responses.items() if v != "Y"]
append_run_markdown_event(
    RUN_DIR,
    heading="Manual notebook gate",
    payload={
        "responses": json.dumps(responses),
        "failed_items": json.dumps(failed),
    },
)

if failed:
    raise AssertionError(f"Notebook gate failed for: {failed}")

print("Notebook manual gate passed.")
